# Dubins Car (3D)

**What you will learn:** build a 3D grid with a **periodic** angle axis, define a
spatial target, and visualize a backward reachable tube.

**pyspect API:** `TVHJImpl`, `reach`

**Prerequisites:** [`reach_avoid.ipynb`](reach_avoid.ipynb)

Backward reachable tube of a Dubins car towards the unit disk at the origin, with
disturbance. State `(x, y, theta)`.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import DubinsCar

In [ ]:
# Same grid as hjr_examples: x, y in [-3,3], theta in [-pi,pi] periodic.
# The horizon is given as positive time: `reach` integrates backwards internally.
AXES = [
    dict(name='t', bounds=[0, np.pi], points=20),
    dict(name='x', bounds=[-3, 3], points=51),
    dict(name='y', bounds=[-3, 3], points=51),
    dict(name='*theta', bounds=[-np.pi, np.pi], points=51),
]

impl = TVHJImpl(dict(cls=DubinsCar), AXES, accuracy='very_high')

In [ ]:
# Target l: unit disk centred at the origin (formula identical to hjr_examples)
l = jnp.linalg.norm(impl.grid.states[..., [0, 1]] - jnp.array([0.0, 0.0]), axis=-1) - 1

In [ ]:
V = impl.reach(l)
print('shape:', V.shape)  # (nt, nx, ny, ntheta)

# V[-1] is the target (zero horizon), V[0] is the tube over the full horizon.
# `ttr[i]` is the time-to-go associated with V[i].
ttr = np.array(impl.timeline)[::-1]

In [ ]:
# Slice at theta ~ 0 (index 25/51 in hjr_examples)
k = V.shape[-1] // 2
print(f'theta = {float(impl.grid.coordinate_vectors[2][k]):.3f} rad')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    np.array(V[0, :, :, k]).T,
    origin='lower',
    extent=[-3, 3, -3, 3],
    aspect='equal',
    cmap='viridis',
)
ax.contour(
    np.array(V[0, :, :, k]).T,
    levels=[0],
    colors='black',
    linewidths=0.8,
    origin='lower',
    extent=[-3, 3, -3, 3],
)
ax.contour(
    np.array(l[:, :, k]).T,
    levels=[0],
    colors='green',
    linewidths=0.8,
    origin='lower',
    extent=[-3, 3, -3, 3],
)
plt.colorbar(im, ax=ax, label=r'$V(x,t)$')
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title(f'Tube over {ttr[0]:.2f} s (target in green)')
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from IPython.display import HTML

# Same rendering as hjr_examples: cyan/white/pink colormap centred on the zero level
V_np = np.array(V[:, :, :, k])

colors = ['cyan', 'white', 'pink']
cmap = LinearSegmentedColormap.from_list('custom_blue_black_white', colors, N=256)
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=np.max(V_np))

EXTENT = [-3, 3, -3, 3]
Xk = np.array(impl.grid.states[..., k, 0])
Yk = np.array(impl.grid.states[..., k, 1])

fig, ax = plt.subplots(figsize=(6, 5))

def update(i):
    ax.clear()
    ax.imshow(V_np[i].T, cmap=cmap, origin='lower', norm=norm,
              extent=EXTENT, aspect='equal')
    ax.contour(Xk.T, Yk.T, V_np[i].T, levels=[0], colors='black', linewidths=1.2)
    ax.contour(Xk.T, Yk.T, np.array(l[:, :, k]).T, levels=[0], colors='green', linewidths=1.2)
    ax.set_xlabel(r'$x_1$')
    ax.set_ylabel(r'$x_2$')
    ax.set_title(f'time-to-go = {ttr[i]:.2f} s')

# Start from the target and walk back in time, like the hjr_examples animation.
frames = list(range(len(V_np) - 1, -1, -1))
ani = FuncAnimation(fig, update, frames=frames, interval=200, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())
